In [1]:
from pathlib import Path
import sys

print("현재 작업 폴더:", Path.cwd())
print("사용 중인 Python:", sys.executable)

현재 작업 폴더: C:\ai_campus_main_proj\data_research\D13
사용 중인 Python: C:\Users\SD1-08\miniforge3\envs\aicam\python.exe


In [3]:
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

ticker = "MSFT"
url = "https://www.sec.gov/files/company_tickers_exchange.json"

email = input("연락 가능한 이메일 입력 후 Enter: ").strip()

if "@" not in email:
    raise ValueError("이메일 주소를 확인하세요.")

request = Request(
    url,
    headers={
        "User-Agent": f"Twodigit Research {email}",
        "Accept": "application/json",
    },
)

print("요청 준비 완료:", ticker)

연락 가능한 이메일 입력 후 Enter:  skkufi0113@skkukdp.re.kr


요청 준비 완료: MSFT


In [4]:
try:
    # 준비한 요청을 SEC에 전송
    with urlopen(request, timeout=30) as response:
        http_status = response.status
        data = json.loads(response.read().decode("utf-8"))

    print("HTTP 상태:", http_status)
    print("응답 필드:", data["fields"])
    print("전체 매핑 행 수:", len(data["data"]))

    # 필드명과 각 행의 값을 연결
    companies = [
        dict(zip(data["fields"], row))
        for row in data["data"]
    ]

    # 티커가 정확히 MSFT인 행 찾기
    matches = [
        company
        for company in companies
        if str(company.get("ticker", "")).upper() == ticker
    ]

    if len(matches) != 1:
        raise ValueError(f"MSFT 검색 결과 {len(matches)}건: 확인 필요")

    company = matches[0]
    cik = str(int(company["cik"])).zfill(10)

    print("\n회사명:", company["name"])
    print("티커:", company["ticker"])
    print("거래소:", company["exchange"])
    print("10자리 CIK:", cik)

except HTTPError as error:
    print("HTTP 오류:", error.code)
except URLError as error:
    print("네트워크 오류:", error.reason)

HTTP 상태: 200
응답 필드: ['cik', 'name', 'ticker', 'exchange']
전체 매핑 행 수: 10445

회사명: MICROSOFT CORP
티커: MSFT
거래소: Nasdaq
10자리 CIK: 0000789019


In [5]:
import random

# 이미 받은 전체 매핑에서 무작위 10행 선택
random_companies = random.sample(companies, k=min(10, len(companies)))

for row in random_companies:
    cik_10 = str(int(row["cik"])).zfill(10)

    print(
        f"티커: {row['ticker']} | "
        f"회사명: {row['name']} | "
        f"거래소: {row['exchange']} | "
        f"CIK: {cik_10}"
    )

티커: FXHO | 회사명: UTime Ltd | 거래소: Nasdaq | CIK: 0001789299
티커: SNPS | 회사명: SYNOPSYS INC | 거래소: Nasdaq | CIK: 0000883241
티커: NTAP | 회사명: NetApp, Inc. | 거래소: Nasdaq | CIK: 0001002047
티커: NPB | 회사명: NORTHPOINTE BANCSHARES INC | 거래소: NYSE | CIK: 0001336706
티커: HRMY | 회사명: Harmony Biosciences Holdings, Inc. | 거래소: Nasdaq | CIK: 0001802665
티커: NLY-PI | 회사명: ANNALY CAPITAL MANAGEMENT INC | 거래소: NYSE | CIK: 0001043219
티커: HOVRW | 회사명: New Horizon Aircraft Ltd. | 거래소: Nasdaq | CIK: 0001930021
티커: EPRX | 회사명: EUPRAXIA PHARMACEUTICALS INC. | 거래소: Nasdaq | CIK: 0001581178
티커: BB | 회사명: BLACKBERRY Ltd | 거래소: NYSE | CIK: 0001070235
티커: OFIX | 회사명: Orthofix Medical Inc. | 거래소: Nasdaq | CIK: 0000884624


In [6]:
import csv
from pathlib import Path

# 이미 받은 전체 데이터에서 MSFT 찾기
msft_rows = [
    row for row in companies
    if str(row.get("ticker", "")).upper() == "MSFT"
]

if len(msft_rows) != 1:
    raise ValueError("MSFT가 정확히 한 행인지 확인이 필요합니다.")

msft = msft_rows[0]

# 저장할 정보
sample = {
    "ticker": msft["ticker"],
    "name": msft["name"],
    "exchange": msft["exchange"],
    "cik": str(int(msft["cik"])).zfill(10),
}

# 현재 D13 폴더에 저장
sample_path = Path.cwd() / "D13_MSFT_sample.csv"

with sample_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=sample.keys())
    writer.writeheader()
    writer.writerow(sample)

print("저장 위치:", sample_path)
print("저장한 데이터:", sample)

저장 위치: C:\ai_campus_main_proj\data_research\D13\D13_MSFT_sample.csv
저장한 데이터: {'ticker': 'MSFT', 'name': 'MICROSOFT CORP', 'exchange': 'Nasdaq', 'cik': '0000789019'}


## D13 검증 결과

- 출처: https://www.sec.gov/files/company_tickers_exchange.json
- HTTP 상태: 200 — 요청 성공
- 전체 매핑: 10,445행
- 응답 필드: cik, name, ticker, exchange
- 확인 기업: MICROSOFT CORP
- 티커: MSFT
- 거래소: Nasdaq
- 10자리 CIK: 0000789019
- 샘플 파일: D13_MSFT_sample.csv

### 확인 범위와 제한
전체 매핑에서 MSFT를 찾아 회사명·티커·CIK·거래소를 확인했다.
현재 매핑이며 과거 티커 변경 이력은 검증하지 않았다.
공시 원문·재무 수치 및 상용·재배포 권한은 아직 검증하지 않았다.

In [7]:
from collections import Counter
from pathlib import Path
import csv

fields = ["cik", "name", "ticker", "exchange"]

# 1. 결측: 필요한 값이 비었는지
print("[1] 필드별 결측")
for field in fields:
    count = sum(
        row.get(field) is None or str(row.get(field)).strip() == ""
        for row in companies
    )
    print(f"{field}: {count}행")

# 2. 완전히 동일한 행이 반복되는지
row_counts = Counter(
    tuple(str(row.get(field, "")) for field in fields)
    for row in companies
)
print("\n[2] 완전 중복 추가 행 수:",
      sum(count - 1 for count in row_counts.values()))

# 3. CIK를 10자리 숫자 문자열로 만들 수 있는지
def valid_cik(value):
    text = str(value).strip()
    return text.isascii() and text.isdigit() and 0 < int(text) < 10**10

invalid_cik = [row for row in companies if not valid_cik(row.get("cik"))]
print("\n[3] CIK 형식 이상:", len(invalid_cik), "행")

# 4. 같은 티커가 여러 CIK에 연결되는지
ticker_to_ciks = {}
for row in companies:
    symbol = str(row.get("ticker", "")).strip().upper()
    ticker_to_ciks.setdefault(symbol, set()).add(str(row.get("cik")))

conflicts = {
    symbol: sorted(ciks)
    for symbol, ciks in ticker_to_ciks.items()
    if symbol and len(ciks) > 1
}
print("\n[4] 여러 CIK에 연결된 티커:", len(conflicts), "개")
print("예시:", list(conflicts.items())[:5])

# 5. MSFT가 정확히 한 행인지
msft_rows = [
    row for row in companies
    if str(row.get("ticker", "")).upper() == "MSFT"
]
print("\n[5] MSFT 일치:", len(msft_rows), "행")
for row in msft_rows:
    print(row)

# 6. 저장한 CSV를 다시 읽어 원래 데이터와 비교
path = Path.cwd() / "D13_MSFT_sample.csv"
with path.open(encoding="utf-8-sig", newline="") as file:
    saved_rows = list(csv.DictReader(file))

if len(msft_rows) == 1:
    row = msft_rows[0]
    expected = {
        "ticker": row["ticker"],
        "name": row["name"],
        "exchange": row["exchange"],
        "cik": str(int(row["cik"])).zfill(10),
    }
    print("\n[6] CSV 내용 일치:", saved_rows == [expected])

[1] 필드별 결측
cik: 0행
name: 0행
ticker: 0행
exchange: 182행

[2] 완전 중복 추가 행 수: 0

[3] CIK 형식 이상: 0 행

[4] 여러 CIK에 연결된 티커: 0 개
예시: []

[5] MSFT 일치: 1 행
{'cik': 789019, 'name': 'MICROSOFT CORP', 'ticker': 'MSFT', 'exchange': 'Nasdaq'}

[6] CSV 내용 일치: True
